# Python for Data Analysis, chapter 7: data cleaning and preparation

The most useful examples from the chapter, rewritten for **pandas 3**, on a messy support-ticket export like the ones you see at work.

**How to use this notebook**
1. Run each cell with **Shift+Enter** and read the output before moving on.
2. Change something in each example and guess the result before running it.
3. The **Your turn** tasks at the end use the real GitHub incidents file. Write those yourself.

Book chapter: https://wesmckinney.com/book/data-cleaning

In [1]:
import io
import json
from pathlib import Path

import numpy as np
import pandas as pd

SAMPLES = Path("../samples")

## The messy data

Look at the raw text before running anything. Spot the problems yourself first: missing values written three different ways, a duplicated row, inconsistent team names, durations with units, an impossible value, a typo.

In [2]:
raw_csv = """ticket_id,opened,team,priority,reporter_email,time_to_close,description
101,2026-11-01 08:15,Network,high,dana@ness.co.il,45 min,VPN down for floor 3
102,2026-11-01 09:30,support ,Low,avi@ness.co.il,N/A,Printer jam
103,2026-11-02 11:00,SUPPORT,medium,noa@client.com,2h,Outlook crash ERR-4021 after update
103,2026-11-02 11:00,SUPPORT,medium,noa@client.com,2h,Outlook crash ERR-4021 after update
104,2026-11-02 14:20,network,hgih,,90 min,Switch port flapping
105,2026-11-03 10:05,Data,low,omer@ness.co.il,-,Report refresh failed ERR-1102
106,2026-11-03 16:40,support,high,maya@client.com,9999 min,Laptop won't boot
107,2026-11-04 08:00,Network,,lior@ness.co.il,30 min,Wi-Fi slow ERR-0007
"""

tickets = pd.read_csv(io.StringIO(raw_csv))
tickets

,ticket_id,opened,team,priority,reporter_email,time_to_close,description
0,101,2026-11-01 08:15,Network,high,dana@ness.co.il,45 min,VPN down for floor 3
1,102,2026-11-01 09:30,support,Low,avi@ness.co.il,NaN,Printer jam
2,103,2026-11-02 11:00,SUPPORT,medium,noa@client.com,2h,Outlook crash ERR-4021 after update
3,103,2026-11-02 11:00,SUPPORT,medium,noa@client.com,2h,Outlook crash ERR-4021 after update
4,104,2026-11-02 14:20,network,hgih,NaN,90 min,Switch port flapping
5,105,2026-11-03 10:05,Data,low,omer@ness.co.il,-,Report refresh failed ERR-1102
6,106,2026-11-03 16:40,support,high,maya@client.com,9999 min,Laptop won't boot
7,107,2026-11-04 08:00,Network,NaN,lior@ness.co.il,30 min,Wi-Fi slow ERR-0007


## 1. Missing data

`read_csv` already turned the empty cells and `N/A` into missing values, but not `-`. Tell it which texts mean "missing" with `na_values`.

In [3]:
tickets = pd.read_csv(io.StringIO(raw_csv), na_values=["N/A", "-", ""])
tickets.isna().sum()     # missing values per column

ticket_id         0
opened            0
team              0
priority          1
reporter_email    1
time_to_close     2
description       0
dtype: int64

In [4]:
# Rows with any missing value
tickets[tickets.isna().any(axis=1)]

,ticket_id,opened,team,priority,reporter_email,time_to_close,description
1,102,2026-11-01 09:30,support,Low,avi@ness.co.il,NaN,Printer jam
4,104,2026-11-02 14:20,network,hgih,NaN,90 min,Switch port flapping
5,105,2026-11-03 10:05,Data,low,omer@ness.co.il,NaN,Report refresh failed ERR-1102
7,107,2026-11-04 08:00,Network,NaN,lior@ness.co.il,30 min,Wi-Fi slow ERR-0007


In [5]:
# Three ways to deal with them; each returns a new table, `tickets` is unchanged
print(len(tickets.dropna()), "rows if we drop every row with a gap")
print(len(tickets.dropna(subset=["priority"])), "rows if we only require a priority")
tickets.fillna({"priority": "unknown", "reporter_email": "unknown"}).head(3)

4 rows if we drop every row with a gap
7 rows if we only require a priority


,ticket_id,opened,team,priority,reporter_email,time_to_close,description
0,101,2026-11-01 08:15,Network,high,dana@ness.co.il,45 min,VPN down for floor 3
1,102,2026-11-01 09:30,support,Low,avi@ness.co.il,NaN,Printer jam
2,103,2026-11-02 11:00,SUPPORT,medium,noa@client.com,2h,Outlook crash ERR-4021 after update


Which one is right depends on the question. Dropping rows silently removes data, so in a pipeline you usually **count** the gaps and keep a record, rather than deleting them.

## 2. Duplicates

In [6]:
tickets.duplicated()     # True for a row that repeats an earlier one exactly

0    False
1    False
2    False
3     True
4    False
5    False
6    False
7    False
dtype: bool

In [7]:
tickets = tickets.drop_duplicates()
len(tickets)

7

In [8]:
# Duplicates by key only: same ticket ID, even if other columns differ
tickets.duplicated(subset=["ticket_id"]).sum()

np.int64(0)

## 3. Cleaning text: the `.str` accessor

`.str` applies a string method to every value in a column at once. No loops.

In [9]:
tickets["team"].unique()     # 'Network', 'support ', 'SUPPORT', 'network', 'Data'...

<ArrowStringArray>
['Network', 'support ', 'SUPPORT', 'network', 'Data', 'support']
Length: 6, dtype: str

In [10]:
tickets["team"] = tickets["team"].str.strip().str.lower()
tickets["team"].value_counts()

team
network    3
support    3
data       1
Name: count, dtype: int64

In [11]:
tickets["priority"] = tickets["priority"].str.lower()
tickets["priority"].unique()     # still has the typo 'hgih'

<ArrowStringArray>
['high', 'low', 'medium', 'hgih', nan]
Length: 5, dtype: str

## 4. Replacing values: `replace` and `map`

`replace` changes only the values you list. `map` translates every value through a dictionary; anything not in it becomes missing.

In [12]:
tickets["priority"] = tickets["priority"].replace({"hgih": "high"})
tickets["priority"].value_counts(dropna=False)

priority
high      3
low       2
medium    1
NaN       1
Name: count, dtype: int64

In [13]:
# map: turn a label into a number for sorting
priority_rank = {"high": 1, "medium": 2, "low": 3}
tickets["priority_rank"] = tickets["priority"].map(priority_rank)
tickets[["ticket_id", "priority", "priority_rank"]]

,ticket_id,priority,priority_rank
0,101,high,1.0
1,102,low,3.0
2,103,medium,2.0
4,104,high,1.0
5,105,low,3.0
6,106,high,1.0
7,107,NaN,NaN


## 5. Regular expressions: pulling structure out of text

`str.extract` keeps the part of the text inside the brackets of a pattern. Two of the most useful patterns for logs and tickets:

In [14]:
# The error code, if the description has one: ERR- followed by 4 digits
tickets["error_code"] = tickets["description"].str.extract(r"(ERR-\d{4})", expand=False)
tickets[["ticket_id", "description", "error_code"]]

,ticket_id,description,error_code
0,101,VPN down for floor 3,NaN
1,102,Printer jam,NaN
2,103,Outlook crash ERR-4021 after update,ERR-4021
4,104,Switch port flapping,NaN
5,105,Report refresh failed ERR-1102,ERR-1102
6,106,Laptop won't boot,NaN
7,107,Wi-Fi slow ERR-0007,ERR-0007


In [15]:
# The email domain: everything after the @
tickets["domain"] = tickets["reporter_email"].str.extract(r"@(.+)$", expand=False)
tickets["domain"].value_counts(dropna=False)

domain
ness.co.il    4
client.com    2
NaN           1
Name: count, dtype: int64

## 6. Turning text into numbers

`time_to_close` mixes units: `45 min`, `2h`. Extract the number and the unit with one pattern, then convert everything to minutes.

In [16]:
parts = tickets["time_to_close"].str.extract(r"(\d+)\s*(min|h)")
parts.columns = ["amount", "unit"]
parts

,amount,unit
0,45,min
1,NaN,NaN
2,2,h
4,90,min
5,NaN,NaN
6,9999,min
7,30,min


In [17]:
amount = pd.to_numeric(parts["amount"])
tickets["minutes_to_close"] = amount.where(parts["unit"] == "min", amount * 60)
tickets[["ticket_id", "time_to_close", "minutes_to_close"]]

,ticket_id,time_to_close,minutes_to_close
0,101,45 min,45.0
1,102,NaN,NaN
2,103,2h,120.0
4,104,90 min,90.0
5,105,NaN,NaN
6,106,9999 min,9999.0
7,107,30 min,30.0


## 7. Outliers

9999 minutes is almost a week for a laptop that won't boot. Probably a placeholder someone typed, not a real duration. Find values outside a sensible range and decide what to do with them: here we mark them as missing rather than delete the ticket.

In [18]:
tickets["minutes_to_close"].describe()

count       5.000000
mean     2056.800000
std      4439.968885
min        30.000000
25%        45.000000
50%        90.000000
75%       120.000000
max      9999.000000
Name: minutes_to_close, dtype: float64

In [19]:
too_long = tickets["minutes_to_close"] > 24 * 60          # more than a day
tickets.loc[too_long, ["ticket_id", "minutes_to_close"]]

,ticket_id,minutes_to_close
6,106,9999.0


In [20]:
tickets.loc[too_long, "minutes_to_close"] = np.nan
tickets["minutes_to_close"].describe()

count      4.000000
mean      71.250000
std       41.306779
min       30.000000
25%       41.250000
50%       67.500000
75%       97.500000
max      120.000000
Name: minutes_to_close, dtype: float64

## 8. Binning: `pd.cut`

Turn a number into ranges, such as SLA buckets.

In [21]:
tickets["sla_bucket"] = pd.cut(
    tickets["minutes_to_close"],
    bins=[0, 60, 240, np.inf],
    labels=["under 1h", "1-4h", "over 4h"],
)
tickets[["ticket_id", "minutes_to_close", "sla_bucket"]]

,ticket_id,minutes_to_close,sla_bucket
0,101,45.0,under 1h
1,102,NaN,NaN
2,103,120.0,1-4h
4,104,90.0,1-4h
5,105,NaN,NaN
6,106,NaN,NaN
7,107,30.0,under 1h


## 9. Types: dates, nullable integers and categories

In [22]:
tickets["opened"] = pd.to_datetime(tickets["opened"])
tickets["minutes_to_close"] = tickets["minutes_to_close"].astype("Int64")   # whole numbers, gaps allowed
tickets["team"] = tickets["team"].astype("category")                       # few repeating labels
tickets.dtypes

ticket_id                    int64
opened              datetime64[us]
team                      category
priority                       str
reporter_email                 str
time_to_close                  str
description                    str
priority_rank              float64
error_code                     str
domain                         str
minutes_to_close             Int64
sla_bucket                category
dtype: object

## 10. Renaming columns

In [23]:
clean = tickets.rename(columns={"reporter_email": "reporter", "time_to_close": "time_to_close_raw"})
clean.columns.tolist()

['ticket_id',
 'opened',
 'team',
 'priority',
 'reporter',
 'time_to_close_raw',
 'description',
 'priority_rank',
 'error_code',
 'domain',
 'minutes_to_close',
 'sla_bucket']

## Your turn

Use the real file `github_incidents.json`. Write these yourself; each needs a few lines.

1. Load it into `incidents` with `json_normalize`, as in chapter 6. How many missing values does each column have? Which columns are missing in **every** row?
2. Are there duplicate incident IDs? Show the count.
3. Make a column `name_clean`: the incident `name` in lowercase with spaces trimmed. Then count how many names contain the word `actions` (hint: `.str.contains`).
4. With `str.extract`, pull the first word after `"with "` from each name, for example `"Incident with Git Operations"` gives `Git`. How many incidents mention each one?
5. Compute `hours_to_resolve`, then bin it with `pd.cut` into `under 1h`, `1-4h`, `over 4h`. How many incidents fall in each bucket?
6. Make `impact` a `category` column with the order `none < minor < major < critical`, and sort the incidents by it, most severe first. (Hint: `pd.CategoricalDtype([...], ordered=True)`.)

In [24]:
with open(SAMPLES / "github_incidents.json") as f:
    raw = json.load(f)

raw.keys()
incidents = pd.json_normalize(raw["incidents"])
incidents.isna().sum()

id                     0
name                   0
status                 0
created_at             0
updated_at             0
monitoring_at         10
resolved_at            0
impact                 0
shortlink              0
started_at             0
page_id                0
incident_updates       0
components             0
reminder_intervals    50
dtype: int64

In [43]:
incidents["id"].duplicated().sum()


np.int64(0)

In [46]:
incidents["name_clean"] = incidents["name"].str.strip().str.lower()
incidents["name_clean"].str.contains("actions").sum()

np.int64(18)

In [58]:
incidents["name"].str.extract(r"with\s(\w+)",expand=False).value_counts(dropna=False)

name
NaN           22
Actions        9
Git            4
some           4
Pull           2
Dependabot     2
Webhooks       1
API            1
GitHub         1
Copilot        1
billing        1
Gemini         1
several        1
Name: count, dtype: int64

In [ ]:
incidents["hours_to_resolve"] = (
    pd.to_datetime(incidents["resolved_at"]) - pd.to_datetime(incidents["created_at"])
).dt.total_seconds() / 3600

hours = pd.cut(incidents["hours_to_resolve"],
               bins=[0, 60, 240, np.inf],
                   labels=["under 1h", "1-4h", "over 4h"],)
hours.value_counts()

hours_to_resolve
under 1h    47
1-4h         0
over 4h      0
Name: count, dtype: int64

In [78]:

order_category = pd.CategoricalDtype(["none","minor","major","critical"],ordered=True)
incidents["impact"]=incidents["impact"].astype(order_category)
incidents["impact"]

0     critical
1        major
2     critical
3        major
4        minor
5        minor
6        minor
7        minor
8        major
9        major
10       major
11       minor
12       minor
13       minor
14       minor
15       minor
16       major
17    critical
18       minor
19    critical
20        none
21       minor
22       minor
23       minor
24       minor
25       minor
26       minor
27    critical
28    critical
29       minor
30       minor
31       minor
32    critical
33    critical
34    critical
35       major
36    critical
37       minor
38       minor
39       minor
40    critical
41       minor
42       minor
43       minor
44       minor
45       major
46       minor
47       minor
48       minor
49    critical
Name: impact, dtype: category
Categories (4, str): ['none' < 'minor' < 'major' < 'critical']

## Notes in my own words

* dropna deletes rows (or columns) that have missing values. fillna fills the gaps with a value I choose, like "unknown" or 0. 
.ffill() copies the value from above, and .bfill() copies from below.
* .str runs a string method on every value in the column at once, without a loop.
* replace vs map: replace changes only the values I list and leaves the rest. map translates every value through a dictionary, and anything not in it becomes NaN.
* Outliers: a value that can't be true (9999 minutes) ruins averages. I set just that cell to NaN instead of deleting the row, so I keep the rest of the ticket's data.